# 🚀 RANST POC - Google Colab Setup

Run de RANST news detection in Google Colab!

## Setup: Clone repo en install

In [ ]:
# Clone the repository
!git clone https://github.com/yourusername/ranst-poc.git
%cd ranst-poc

In [ ]:
# Run setup script
!python colab_setup.py

## ⚙️ Configure HuggingFace Token

In [ ]:
# TODO: Get your token from https://huggingface.co/settings/tokens

import os

HF_TOKEN = "your_token_here"  # Replace with your token!

# Save to .env
with open('.env', 'w') as f:
    f.write(f"""HF_TOKEN={HF_TOKEN}
OLLAMA_MODEL=mistral
OLLAMA_BASE_URL=http://localhost:11434
NEWS_SCORE_THRESHOLD=0.65
""")

os.environ['HF_TOKEN'] = HF_TOKEN
print("✓ HuggingFace token configured")

## 🦙 Start Ollama Service

In [ ]:
# Start Ollama in the background
!bash /content/start_ollama.sh &

In [ ]:
# Wait for Ollama to start (be patient!)
import time
import requests

print("⏳ Waiting for Ollama to start...")

for i in range(60):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=2)
        if r.status_code == 200:
            print("\n✅ Ollama is running!")
            break
    except:
        pass
    
    print(f".", end="", flush=True)
    time.sleep(1)
else:
    print("\n❌ Ollama startup timeout - try restarting")

## 📺 Download & Analyze Latest Ranst Video

In [ ]:
# Run the analysis
# This will automatically find and download the latest Ranst video
!python main.py

## 📊 View Results

In [ ]:
import json
import os
from pathlib import Path

# Find output files
output_dir = Path('/content/output')
output_files = sorted(output_dir.glob('ranst_news_*.json'))

if not output_files:
    print("❌ No output files found")
else:
    latest = output_files[-1]
    print(f"\n📁 Latest result: {latest.name}\n")
    
    with open(latest, 'r') as f:
        data = json.load(f)
    
    # Show metadata
    meta = data['metadata']
    print(f"📝 Audio file: {meta['audio_file']}")
    print(f"⏱️  Duration: {meta['total_duration_seconds']:.0f}s ({meta['total_duration_seconds']/60:.1f} min)")
    print(f"🗣️  Speakers: {meta['speakers_count']}")
    print(f"📄 Segments: {meta['segments_count']}")
    
    # Show news alerts
    alerts = data['news_alerts']
    print(f"\n🚨 NEWS ALERTS: {len(alerts)} found\n")
    
    for i, alert in enumerate(alerts, 1):
        print(f"{i}. {alert['chunk_time']}")
        print(f"   📊 Score: {alert['score']} | Category: {alert['category']}")
        print(f"   💬 Reason: {alert['reason']}")
        print(f"   📄 Preview: {alert['text_preview'][:100]}...")
        print()

## 🔍 Full Transcript View

In [ ]:
import pandas as pd

# Show transcript as table
transcript = data['full_transcript']
df = pd.DataFrame(transcript[:20])  # First 20 segments

display(df)

## 💾 Download Results

In [ ]:
# Download the result file
from google.colab import files

files.download(str(latest))
print(f"✓ Downloaded: {latest.name}")